In [3]:
%cd ..


/home/maxim/projects/sandbox/cloud-ivan


In [1]:
!pip install boto3 redis rq


  Using cached click-8.3.1-py3-none-any.whl.metadata (2.6 kB)
  Using cached pytz-2025.2-py2.py3-none-any.whl.metadata (22 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.6/14.6 MB 12.7 MB/s eta 0:00:00a 0:00:01
Using cached click-8.3.1-py3-none-any.whl (108 kB)
Using cached pytz-2025.2-py2.py3-none-any.whl (509 kB)

[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [5]:
import os
import time
import boto3
from pathlib import Path
from redis import Redis
from rq import Queue

# 1. Настройки инфраструктуры (внешние, так как мы запускаем из Jupyter)
MINIO_ENDPOINT = "http://localhost:9000"
MINIO_ACCESS = "admin"
MINIO_SECRET = "password123"
BUCKET_NAME = "media-bucket"
REDIS_URL = "redis://localhost:6379/0"

# Пути к тестовым данным
BASE_DIR = Path.cwd()
TEST_VIDEO_PATH = BASE_DIR / "data" / "test" / "vid" / "1.mp4"
DOWNLOAD_RESULT_PATH = BASE_DIR / "data" / "results" / "final_worker_result.mp4"

def test_async_worker():
    print("[*] Инициализация клиентов...")
    # Настраиваем S3 Клиент
    s3 = boto3.client('s3', endpoint_url=MINIO_ENDPOINT, 
                      aws_access_key_id=MINIO_ACCESS, aws_secret_access_key=MINIO_SECRET)
    
    # Создаем бакет, если его нет
    try:
        s3.head_bucket(Bucket=BUCKET_NAME)
    except Exception:
        s3.create_bucket(Bucket=BUCKET_NAME)

    # Настраиваем RQ Клиент
    redis_conn = Redis.from_url(REDIS_URL)
    queue = Queue('media_tasks', connection=redis_conn)

    # 1. Имитация загрузки файла юзером
    input_object_name = f"input_{TEST_VIDEO_PATH.name}"
    output_object_name = f"output_yolo_{TEST_VIDEO_PATH.name}"
    
    print(f"[*] 1. Загрузка исходного видео в MinIO: {input_object_name}")
    s3.upload_file(str(TEST_VIDEO_PATH), BUCKET_NAME, input_object_name)

    # 2. Отправка задачи в очередь воркеру
    print("[*] 2. Постановка задачи в очередь Redis...")
    # Мы передаем путь к функции строкой 'worker.process_media_task', 
    # чтобы RQ знал, что искать внутри контейнера воркера
    job = queue.enqueue(
        'worker.process_media_task', 
        args=(input_object_name, output_object_name, 'yolo'), # Попробуем yolo (или 'pixelate', 'canny')
        job_timeout=600 # Даем воркеру 10 минут на тяжелое видео
    )
    print(f"  [+] Задача отправлена! ID: {job.id}")

    # 3. Асинхронное ожидание (Polling)
    print("[*] 3. Ожидание завершения работы воркера...")
    while not job.is_finished:
        if job.is_failed:
            print(f"\n[!] ОШИБКА ВОРКЕРА. Проверь логи контейнера: docker logs yolo_worker")
            return
        print(".", end="", flush=True)
        time.sleep(2)
        job.refresh() # Обновляем статус задачи из Redis

    print(f"\n[+] Воркер завершил задачу. Результат: {job.result}")

    # 4. Скачивание результата
    print(f"[*] 4. Скачивание готового файла из MinIO...")
    DOWNLOAD_RESULT_PATH.parent.mkdir(parents=True, exist_ok=True)
    s3.download_file(BUCKET_NAME, output_object_name, str(DOWNLOAD_RESULT_PATH))
    print(f"[+] УСПЕХ! Проверь файл: {DOWNLOAD_RESULT_PATH}")

# ЗАПУСК!
test_async_worker()


[*] Инициализация клиентов...
[*] 1. Загрузка исходного видео в MinIO: input_1.mp4
[*] 2. Постановка задачи в очередь Redis...
  [+] Задача отправлена! ID: c966fb6b-de6b-407a-a3bc-57f9e186b789
[*] 3. Ожидание завершения работы воркера...
...............
[+] Воркер завершил задачу. Результат: {'status': 'success', 'result_file': 'output_yolo_1.mp4'}
[*] 4. Скачивание готового файла из MinIO...
[+] УСПЕХ! Проверь файл: /home/maxim/projects/sandbox/cloud-ivan/data/results/final_worker_result.mp4
